In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MCC_ETL")
    .master("local[8]")  # use 8 cores to leave some headroom for OS
    .config("spark.driver.memory", "12g")
    .config("spark.executor.memory", "12g")
    .config("spark.sql.shuffle.partitions", "16")  # number of parallel shuffle tasks
    .config("spark.default.parallelism", "16")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")


25/08/28 04:17:42 WARN Utils: Your hostname, Js-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
25/08/28 04:17:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/28 04:17:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.6


In [2]:
# File path from notebooks directory
file_path = "../data/raw/mcc_raw.csv"

# Load CSV
df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

# Preview
df.printSchema()
df.show(5, truncate=100)


root
 |-- _c0: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- cik: integer (nullable = true)
 |-- company.name: string (nullable = true)
 |-- form.type: string (nullable = true)
 |-- date.filed: date (nullable = true)
 |-- edgar.link: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- index.link: string (nullable = true)
 |-- contract.link: string (nullable = true)
 |-- exhibit: string (nullable = true)
 |-- description: string (nullable = true)
 |-- exhibit_lead: string (nullable = true)
 |-- contract: string (nullable = true)
 |-- type_label: string (nullable = true)
 |-- type_score: string (nullable = true)
 |-- amend: string (nullable = true)
 |-- restate: string (nullable = true)
 |-- joinder: double (nullable = true)
 |-- termination: integer (nullable = true)
 |-- parties: string (nullable = true)
 |-- agreement_type: string (nullable = true)
 |-- parties_cleaned: string (nullable = true)
 |-- master_parties: string (nullable = true)

+---

25/08/28 04:17:50 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , year, cik, company.name, form.type, date.filed, edgar.link, quarter, index.link, contract.link, exhibit, description, exhibit_lead, contract, type_label, type_score, amend, restate, joinder, termination, parties, agreement_type, parties_cleaned, master_parties
 Schema: _c0, year, cik, company.name, form.type, date.filed, edgar.link, quarter, index.link, contract.link, exhibit, description, exhibit_lead, contract, type_label, type_score, amend, restate, joinder, termination, parties, agreement_type, parties_cleaned, master_parties
Expected: _c0 but found: 
CSV file: file:///Users/jefferyjapheth/Documents/dev/devsuite/project/contract_nlp/data/raw/mcc_raw.csv


In [3]:
from pyspark.sql.functions import col

df = df.withColumnRenamed("company.name", "company_name") \
       .withColumnRenamed("form.type", "form_type") \
       .withColumnRenamed("date.filed", "date_filed")

selected_columns = [
    "year",
    "company_name",
    "form_type",
    "date_filed",
    "description",
    "contract",
    "type_label",
    "type_score",
    "amend",
    "restate",
    "joinder",
    "termination",
    "agreement_type"
]

df = df.select(*selected_columns)
df.show(5, truncate=100)


+----+-----------------------+---------+----------+------------------------+----------------------------------------------------------------------------+----------+------------------+-----+-------+-------+-----------+--------------+
|year|           company_name|form_type|date_filed|             description|                                                                    contract|type_label|        type_score|amend|restate|joinder|termination|agreement_type|
+----+-----------------------+---------+----------+------------------------+----------------------------------------------------------------------------+----------+------------------+-----+-------+-------+-----------+--------------+
|2000|STOCKWALK COM GROUP INC|     10-Q|2000-02-14|asset purchase agreement|/Archives/edgar/data/1001136/000095012400000624/0000950124-00-000624-d2.html|   LABEL_4|0.9999955892562866|    0|      0|    0.0|          0|   purchase&ma|
|2000|STOCKWALK COM GROUP INC|     10-Q|2000-02-14| amendment to emp

In [4]:
df.createOrReplaceTempView("contracts")

spark.sql("""
    SELECT type_label, COUNT(*) AS frequency
    FROM contracts
    WHERE type_label IS NOT NULL
    GROUP BY type_label
    ORDER BY frequency DESC
""").show()



+--------------------+---------+
|          type_label|frequency|
+--------------------+---------+
|             LABEL_1|   486331|
|             LABEL_0|   362605|
|             LABEL_4|   141657|
|             LABEL_3|    89836|
|             LABEL_5|    55881|
|             LABEL_6|    46146|
|             LABEL_2|    36707|
|             LABEL_7|    34898|
|               EX-10|       18|
|/Archives/edgar/d...|        2|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|               2000"|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
+--------------------+---------+
only showing top 20 rows



In [9]:
# First, let's create a temp view from your previous query
cleaned_data_df = spark.sql("""
-- Step 1: Identify valid labels with frequency > 10,000
WITH valid_labels AS (
 SELECT type_label
 FROM contracts
 WHERE type_label IS NOT NULL
 AND type_label NOT LIKE '%/%' -- Remove file path entries
 AND type_label NOT LIKE '%edgar%' -- Remove edgar path entries
 AND type_label NOT LIKE '%"' -- Remove malformed entries
 GROUP BY type_label
 HAVING COUNT(*) >= 10000
),
-- Step 2: Clean and deduplicate the main dataset
cleaned_contracts AS (
 SELECT DISTINCT
 c.year,
 c.company_name,
 c.form_type,
 c.date_filed,
 c.description,
 c.contract,
 c.type_label,
 c.type_score,
 c.amend,
 c.restate,
 c.joinder,
 c.termination,
 c.agreement_type,
 -- Add row number for potential further deduplication
 ROW_NUMBER() OVER (
 PARTITION BY c.company_name, c.type_label, c.description, c.contract
 ORDER BY c.date_filed DESC, c.type_score DESC
 ) as rn
 FROM contracts c
 INNER JOIN valid_labels v ON c.type_label = v.type_label
 WHERE c.type_label IS NOT NULL
 AND c.description IS NOT NULL
 AND c.contract IS NOT NULL
 AND LENGTH(TRIM(c.description)) > 0
 AND LENGTH(TRIM(c.contract)) > 0
),
-- Step 3: Final dataset with partitioning info
final_dataset AS (
 SELECT
 *,
 -- Add partitioning columns for balanced sampling
 NTILE(5) OVER (PARTITION BY type_label ORDER BY RAND()) as data_split,
 COUNT(*) OVER (PARTITION BY type_label) as label_count
 FROM cleaned_contracts
 WHERE rn = 1 -- Keep only the most recent/highest scored duplicate
)
-- Step 4: Final selection with your chosen columns
SELECT
 year,
 company_name,
 form_type,
 date_filed,
 description,
 contract,
 type_label,
 type_score,
 amend,
 restate,
 joinder,
 termination,
 agreement_type,
 data_split,
 label_count
FROM final_dataset
ORDER BY type_label, data_split
""")

cleaned_data_df.show(20)



+----+--------------------+---------+----------+--------------------+--------------------+----------+------------------+-----+-------+-------+-----------+------------------+----------+-----------+
|year|        company_name|form_type|date_filed|         description|            contract|type_label|        type_score|amend|restate|joinder|termination|    agreement_type|data_split|label_count|
+----+--------------------+---------+----------+--------------------+--------------------+----------+------------------+-----+-------+-------+-----------+------------------+----------+-----------+
|2012|SMSA Treemont Acq...|     10-Q|2012-05-15|            ex-10.11|/Archives/edgar/d...|   LABEL_0|0.9999231100082396|    0|      0|    0.0|          0|          security|         1|     329945|
|2013|HII Technologies ...|      8-K|2013-07-01|equipment securit...|/Archives/edgar/d...|   LABEL_0|0.9997053742408752|    0|      0|    0.0|          0|          security|         1|     329945|
|2009|TRW AUTOM

In [10]:
# Create temp view
cleaned_data_df.createOrReplaceTempView("cleaned_data")

# Now sample 50 rows from each type_label
spark.sql("""
WITH sampled_data AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY type_label ORDER BY RAND()) as sample_rn
    FROM cleaned_data
)
SELECT 
    contract,
    type_label,
    description,
    agreement_type,
    form_type
FROM sampled_data
WHERE sample_rn <= 50
ORDER BY type_label, sample_rn
""").show(1000, truncate=False)

+--------------------------------------------------------------------------------+----------+--------------------------------------------------------------------------------+------------------------------------+---------+
|contract                                                                        |type_label|description                                                                     |agreement_type                      |form_type|
+--------------------------------------------------------------------------------+----------+--------------------------------------------------------------------------------+------------------------------------+---------+
|/Archives/edgar/data/932372/000119312506172622/dex1036.htm                      |LABEL_0   |promissory note                                                                 |security                            |10-Q     |
|/Archives/edgar/data/1347613/000110465914016491/a14-7414_2ex10d1.htm            |LABEL_0   |ex-10.1            

In [25]:
spark.sql("""
WITH cleaned_contract AS (
    SELECT 
        -- Clean the contract by removing the /Archives/edgar/data/ prefix
        CASE 
            WHEN contract LIKE '/Archives/edgar/data/%' 
            THEN REGEXP_REPLACE(contract, '^/Archives/edgar/data/', '')
            ELSE contract
        END as contract,
        type_label,
        description,
        agreement_type,
        form_type
    FROM cleaned_data
),

-- Add uniqueness check
contract_analysis AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY type_label ORDER BY RAND()) as sample_rn,
        COUNT(*) OVER (PARTITION BY contract) as contract_frequency
    FROM cleaned_contract
)

SELECT 
    contract,
    type_label,
    description,
    agreement_type,
    form_type,
    contract_frequency
FROM contract_analysis
WHERE sample_rn <= 50
ORDER BY type_label, contract_frequency DESC
""").show(1000, truncate=False)

+-----------------------------------------------------------+----------+--------------------------------------------------------------------------------+-------------------------------------------+---------+------------------+
|contract                                                   |type_label|description                                                                     |agreement_type                             |form_type|contract_frequency|
+-----------------------------------------------------------+----------+--------------------------------------------------------------------------------+-------------------------------------------+---------+------------------+
|718916/000119312506205456/dex107.htm                       |LABEL_0   |revolving credit note in favor of pnc bank, national association                |security                                   |S-4      |135               |
|1039508/000095013407014662/d47804exv10w2.htm               |LABEL_0   |exhibit 10.2        

In [35]:
spark.sql("""
SELECT 
    CASE 
        WHEN contract LIKE '/Archives/edgar/data/%' 
        THEN REGEXP_REPLACE(contract, '^/Archives/edgar/data/', '')
        ELSE contract
    END as contract,
    COUNT(*) as frequency,
    COUNT(DISTINCT type_label) as distinct_labels
FROM cleaned_data
GROUP BY contract
ORDER BY frequency DESC
""").show(100, truncate=False)

+------------------------------------------------+---------+---------------+
|contract                                        |frequency|distinct_labels|
+------------------------------------------------+---------+---------------+
|787030/000119312515322536/d941734dex1050.htm    |445      |1              |
|787030/000119312515322536/d941734dex1044.htm    |445      |1              |
|812191/000119312514442695/d829606dex1042.htm    |303      |1              |
|812191/000119312514442695/d829606dex1043.htm    |303      |1              |
|812191/000119312514442695/d829606dex1013.htm    |303      |1              |
|801793/000089322003001682/w89896exv10w75.txt    |254      |1              |
|801793/000089322003001682/w89896exv10w76.txt    |254      |1              |
|801793/000089322003001682/w89896exv2w5.txt      |254      |1              |
|801793/000089322003001682/w89896exv10w74.txt    |254      |1              |
|801793/000093066101501041/dex1049.txt           |232      |1              |

In [33]:
spark.sql("""
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT contract) as unique_contracts,
    ROUND(COUNT(DISTINCT contract) * 100.0 / COUNT(*), 2) as uniqueness_percentage
FROM cleaned_data
""").show()

+----------+----------------+---------------------+
|total_rows|unique_contracts|uniqueness_percentage|
+----------+----------------+---------------------+
|   1156162|          945664|                81.79|
+----------+----------------+---------------------+



In [34]:
spark.sql("""
-- Check if we still have duplicates after the ROW_NUMBER() filter
SELECT
 contract,
 type_label,
 COUNT(*) as occurrence_count
FROM cleaned_contracts
GROUP BY contract, type_label
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC
LIMIT 20
""").show()

+--------------------+----------+----------------+
|            contract|type_label|occurrence_count|
+--------------------+----------+----------------+
|1088283/000119312...|   LABEL_7|               3|
+--------------------+----------+----------------+



In [36]:
final_cleaned_data = spark.sql("""
-- Step 1: Identify valid labels with frequency > 10,000
WITH valid_labels AS (
    SELECT type_label
    FROM contracts
    WHERE type_label IS NOT NULL
      AND type_label NOT LIKE '%/%'
      AND type_label NOT LIKE '%edgar%'
      AND type_label NOT LIKE '%"'
    GROUP BY type_label
    HAVING COUNT(*) >= 10000
),

-- Step 2: Clean and deduplicate the main dataset
cleaned_contracts AS (
    SELECT DISTINCT
        c.year,
        c.company_name,
        c.form_type,
        c.date_filed,
        c.description,
        -- Clean the contract column by removing /Archives/edgar/data/ prefix
        CASE
            WHEN c.contract LIKE '/Archives/edgar/data/%'
            THEN REGEXP_REPLACE(c.contract, '^/Archives/edgar/data/', '')
            ELSE c.contract
        END as contract,
        c.type_label,
        c.type_score,
        c.amend,
        c.restate,
        c.joinder,
        c.termination,
        c.agreement_type,
        -- Add row number for deduplication by contract content
        ROW_NUMBER() OVER (
            PARTITION BY 
                CASE
                    WHEN c.contract LIKE '/Archives/edgar/data/%'
                    THEN REGEXP_REPLACE(c.contract, '^/Archives/edgar/data/', '')
                    ELSE c.contract
                END,
                c.type_label,
                c.description
            ORDER BY c.date_filed DESC, c.type_score DESC
        ) as rn
    FROM contracts c
    INNER JOIN valid_labels v ON c.type_label = v.type_label
    WHERE c.type_label IS NOT NULL
      AND c.description IS NOT NULL
      AND c.contract IS NOT NULL
      AND LENGTH(TRIM(c.description)) > 0
      AND LENGTH(TRIM(c.contract)) > 0
),

-- Step 3: Final dataset with partitioning info
final_dataset AS (
    SELECT 
        *,
        -- Add partitioning columns for balanced sampling
        NTILE(5) OVER (PARTITION BY type_label ORDER BY RAND()) as data_split,
        COUNT(*) OVER (PARTITION BY type_label) as label_count
    FROM cleaned_contracts
    WHERE rn = 1  -- Keep only one instance per contract-type_label-description combination
)

-- Step 4: Final selection
SELECT
    year,
    company_name,
    form_type,
    date_filed,
    description,
    contract,
    type_label,
    type_score,
    amend,
    restate,
    joinder,
    termination,
    agreement_type,
    data_split,
    label_count
FROM final_dataset
ORDER BY type_label, data_split
""")

# Check the final results
final_cleaned_data.createOrReplaceTempView("final_cleaned_data")
spark.sql("""
SELECT 
    COUNT(*) as final_total_rows,
    COUNT(DISTINCT contract) as final_unique_contracts,
    ROUND(COUNT(DISTINCT contract) * 100.0 / COUNT(*), 2) as final_uniqueness_percentage
FROM final_cleaned_data
""").show()

+----------------+----------------------+---------------------------+
|final_total_rows|final_unique_contracts|final_uniqueness_percentage|
+----------------+----------------------+---------------------------+
|          945664|                945664|                     100.00|
+----------------+----------------------+---------------------------+



In [37]:
# Create temp view from your final cleaned data
final_cleaned_data.createOrReplaceTempView("final_cleaned_data")

# Sample 50 rows from each type_label
spark.sql("""
WITH sampled_data AS (
   SELECT *,
       ROW_NUMBER() OVER (PARTITION BY type_label ORDER BY RAND()) as sample_rn
   FROM final_cleaned_data
)
SELECT 
   contract,
   type_label,
   description,
   agreement_type,
   form_type
FROM sampled_data
WHERE sample_rn <= 50
ORDER BY type_label, sample_rn
""").show(1000, truncate=False)

+-----------------------------------------------------------+----------+--------------------------------------------------------------------------------+-----------------------+---------+
|contract                                                   |type_label|description                                                                     |agreement_type         |form_type|
+-----------------------------------------------------------+----------+--------------------------------------------------------------------------------+-----------------------+---------+
|923808/000092290704000317/form10qexh109_051404.htm         |LABEL_0   |exhibit 10.9                                                                    |security               |10-Q     |
|1078547/000110465904029139/a04-11078_1ex10d1.htm           |LABEL_0   |ex-10.1                                                                         |security               |8-K      |
|930797/000109489101500289/kirlin_91101-exh1017.txt         

In [38]:
# Check text quality and length distribution
spark.sql("""
SELECT 
    type_label,
    AVG(LENGTH(description)) as avg_description_length,
    AVG(LENGTH(contract)) as avg_contract_length,
    COUNT(*) as sample_count
FROM final_cleaned_data 
GROUP BY type_label
ORDER BY type_label
""").show()

+----------+----------------------+-------------------+------------+
|type_label|avg_description_length|avg_contract_length|sample_count|
+----------+----------------------+-------------------+------------+
|   LABEL_0|    23.672967242298142|  44.24591520869003|      264579|
|   LABEL_1|    24.310020381674217|  44.36530111186985|      383678|
|   LABEL_2|    21.349738016689308|   44.5787696487483|       25765|
|   LABEL_3|     24.72389958842565|  44.56464230732979|       63658|
|   LABEL_4|    23.870911019061516| 43.763330825636515|      110327|
|   LABEL_5|    23.955817642243144|  44.70113589848547|       39088|
|   LABEL_6|     23.57875255027689| 44.378373651996505|       34310|
|   LABEL_7|     25.37260398202729|   43.0453439960427|       24259|
+----------+----------------------+-------------------+------------+



In [39]:
# Check text quality and sample content
spark.sql("""
SELECT 
    type_label,
    AVG(LENGTH(description)) as avg_description_length,
    MIN(LENGTH(description)) as min_description_length,
    MAX(LENGTH(description)) as max_description_length,
    AVG(LENGTH(contract)) as avg_contract_length,
    MIN(LENGTH(contract)) as min_contract_length,
    MAX(LENGTH(contract)) as max_contract_length,
    COUNT(*) as sample_count
FROM final_cleaned_data 
GROUP BY type_label
ORDER BY type_label
""").show()

+----------+----------------------+----------------------+----------------------+-------------------+-------------------+-------------------+------------+
|type_label|avg_description_length|min_description_length|max_description_length|avg_contract_length|min_contract_length|max_contract_length|sample_count|
+----------+----------------------+----------------------+----------------------+-------------------+-------------------+-------------------+------------+
|   LABEL_0|    23.672967242298142|                     1|                    84|  44.24591520869003|                 30|                 59|      264579|
|   LABEL_1|    24.310020381674217|                     1|                    84|  44.36530111186985|                 31|                 59|      383678|
|   LABEL_2|    21.349738016689308|                     1|                    80|   44.5787696487483|                 32|                 59|       25765|
|   LABEL_3|     24.72389958842565|                     2|            

In [40]:
# Sample actual text content from each label
spark.sql("""
WITH text_samples AS (
    SELECT 
        type_label,
        description,
        contract,
        agreement_type,
        ROW_NUMBER() OVER (PARTITION BY type_label ORDER BY RAND()) as rn
    FROM final_cleaned_data
    WHERE LENGTH(TRIM(description)) > 20  -- Filter out very short descriptions
)
SELECT 
    type_label,
    SUBSTRING(description, 1, 200) as description_sample,
    SUBSTRING(contract, 1, 100) as contract_sample,
    agreement_type
FROM text_samples
WHERE rn <= 5  -- 5 samples per label
ORDER BY type_label, rn
""").show(1000, truncate=False)

+----------+--------------------------------------------------------------------------------+---------------------------------------------------------+---------------+
|type_label|description_sample                                                              |contract_sample                                          |agreement_type |
+----------+--------------------------------------------------------------------------------+---------------------------------------------------------+---------------+
|LABEL_0   |amend. #1 to loan documents, dated 08/21/2003                                   |225263/000119312503080479/dex101.htm                     |security       |
|LABEL_0   |ex-10.2: loan agreement                                                         |1289788/000095012305013524/y14612exv10w2.htm             |security       |
|LABEL_0   |form of subscription agreement                                                  |842013/000121390019007495/f8k041219ex10-1_hashlabsinc.htm|security 

In [43]:
# Check for empty or low-quality text
spark.sql("""
SELECT 
    type_label,
    COUNT(*) as total_samples,
    SUM(CASE WHEN LENGTH(TRIM(description)) <= 10 THEN 1 ELSE 0 END) as very_short_descriptions,
    SUM(CASE WHEN description LIKE '%null%' OR description LIKE '%N/A%' THEN 1 ELSE 0 END) as null_like_descriptions,
    SUM(CASE WHEN LENGTH(TRIM(contract)) <= 10 THEN 1 ELSE 0 END) as very_short_contracts,
    ROUND(AVG(SIZE(SPLIT(description, ' '))), 2) as avg_words_in_description
FROM final_cleaned_data
GROUP BY type_label
ORDER BY type_label
""").show()

+----------+-------------+-----------------------+----------------------+--------------------+------------------------+
|type_label|total_samples|very_short_descriptions|null_like_descriptions|very_short_contracts|avg_words_in_description|
+----------+-------------+-----------------------+----------------------+--------------------+------------------------+
|   LABEL_0|       264579|                  71968|                     0|                   0|                    3.49|
|   LABEL_1|       383678|                 118053|                     0|                   0|                    3.49|
|   LABEL_2|        25765|                   8124|                     0|                   0|                    3.22|
|   LABEL_3|        63658|                  17458|                     0|                   0|                    3.44|
|   LABEL_4|       110327|                  30561|                     0|                   0|                    3.59|
|   LABEL_5|        39088|              

In [44]:
spark.sql("""
SELECT type_label, description, LENGTH(description) as desc_length
FROM final_cleaned_data 
WHERE LENGTH(TRIM(description)) > 50
ORDER BY type_label, RAND()
LIMIT 20
""").show(truncate=False)

+----------+--------------------------------------------------------------------------------+-----------+
|type_label|description                                                                     |desc_length|
+----------+--------------------------------------------------------------------------------+-----------+
|LABEL_0   |amendment to amended and restated receivables purchase agreement                |64         |
|LABEL_0   |letter agreement between wrc media inc. and eac iii l.l.c.                      |58         |
|LABEL_0   |third amendment to securities purchanse agreement and subordination             |67         |
|LABEL_0   |ex-10.3 form of second amendment to credit agreement                            |52         |
|LABEL_0   |second amendment to limited waiver to the transfer and administration agreement |79         |
|LABEL_0   |ex-10.1 form of commitment letter dated april 25, 2005                          |54         |
|LABEL_0   |securities purchase agreement with

In [45]:
# Create a filtered dataset with only meaningful descriptions
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW filtered_contracts AS
SELECT *
FROM final_cleaned_data
WHERE LENGTH(TRIM(description)) > 50
""")

DataFrame[]

In [46]:
# Check the filtered dataset quality
spark.sql("""
SELECT 
    type_label,
    COUNT(*) as filtered_samples,
    AVG(LENGTH(description)) as avg_desc_length,
    AVG(SIZE(SPLIT(description, ' '))) as avg_word_count
FROM filtered_contracts
GROUP BY type_label
ORDER BY type_label
""").show()

+----------+----------------+-----------------+------------------+
|type_label|filtered_samples|  avg_desc_length|    avg_word_count|
+----------+----------------+-----------------+------------------+
|   LABEL_0|           28797|68.11511615793312|10.412195714831405|
|   LABEL_1|           50685|65.93175495708789| 9.541166025451316|
|   LABEL_2|            2140|68.44579439252337|10.602803738317757|
|   LABEL_3|            8105| 68.5305367057372| 9.875262183837137|
|   LABEL_4|           12195|67.59229192291923|10.469372693726937|
|   LABEL_5|            4572|70.27668416447943| 10.61395450568679|
|   LABEL_6|            3825|68.99581699346405| 9.965751633986928|
|   LABEL_7|            3100|68.55322580645161|10.231935483870968|
+----------+----------------+-----------------+------------------+



In [47]:
# Sample from other labels to understand what each represents
spark.sql("""
SELECT type_label, description, LENGTH(description) as desc_length
FROM filtered_contracts
WHERE type_label IN ('LABEL_1', 'LABEL_2', 'LABEL_3', 'LABEL_4')
ORDER BY type_label, RAND()
LIMIT 20
""").show(truncate=False)

+----------+--------------------------------------------------------------------------------+-----------+
|type_label|description                                                                     |desc_length|
+----------+--------------------------------------------------------------------------------+-----------+
|LABEL_1   |amendment no. 2 dated february 1, 2003, to deferred compensation plan           |69         |
|LABEL_1   |form of notice of grant of stock options and option exercise and stock purchase |79         |
|LABEL_1   |amended and restated management continuity agreement of g. william beale        |72         |
|LABEL_1   |amendment no. 1 to the third amended and restated employment agreement          |70         |
|LABEL_1   |ex-10.1 form of restricted stock unit agreement - non-employee directors 2022   |77         |
|LABEL_1   |form of 2012 resticted stock unit agreement for non-executive employees under th|80         |
|LABEL_1   |employment agreement between pacer

In [51]:
# Check remaining labels
spark.sql("""
SELECT type_label, description, LENGTH(description) as desc_length
FROM filtered_contracts
WHERE type_label IN ('LABEL_2', 'LABEL_3', 'LABEL_4', 'LABEL_5', 'LABEL_6', 'LABEL_7')
ORDER BY type_label, RAND()
LIMIT 30
""").show(truncate=False)

+----------+--------------------------------------------------------------------------------+-----------+
|type_label|description                                                                     |desc_length|
+----------+--------------------------------------------------------------------------------+-----------+
|LABEL_2   |exhibit 10.110 lease for 48 rue des francs-bourgeois (french original)          |70         |
|LABEL_2   |lease for the property located in valparaiso, indiana                           |53         |
|LABEL_2   |first amendment to lease agreement between purple innovation, llc and pnk s2, ll|80         |
|LABEL_2   |lease agreement dated december 15, 2005, between the bank and great meadows inc.|80         |
|LABEL_2   |oil sands lease no. 7406080084 dated august 10, 2006                            |52         |
|LABEL_2   |ex-10.35 first amend. dated december 13, 1996 to lease agreement                |64         |
|LABEL_2   |sublease between modtech and boise

In [49]:
# Check how much data remains after filtering
spark.sql("""
SELECT 
    type_label,
    COUNT(*) as filtered_samples,
    AVG(LENGTH(description)) as avg_desc_length,
    AVG(SIZE(SPLIT(description, ' '))) as avg_word_count
FROM filtered_contracts
GROUP BY type_label
ORDER BY type_label
""").show()

+----------+----------------+-----------------+------------------+
|type_label|filtered_samples|  avg_desc_length|    avg_word_count|
+----------+----------------+-----------------+------------------+
|   LABEL_0|           28797|68.11511615793312|10.412195714831405|
|   LABEL_1|           50685|65.93175495708789| 9.541166025451316|
|   LABEL_2|            2140|68.44579439252337|10.602803738317757|
|   LABEL_3|            8105| 68.5305367057372| 9.875262183837137|
|   LABEL_4|           12195|67.59229192291923|10.469372693726937|
|   LABEL_5|            4572|70.27668416447943| 10.61395450568679|
|   LABEL_6|            3825|68.99581699346405| 9.965751633986928|
|   LABEL_7|            3100|68.55322580645161|10.231935483870968|
+----------+----------------+-----------------+------------------+



In [ ]:
spark.stop()